# IEEE-CIS Fraud Detection: 비용 민감형 사기탐지 모델 개발
## Day 7. 확률 보정(Calibration) + Threshold Optimization

**목표**
- LightGBM(튜닝), RF(기본값) 두 최종 후보에 CalibratedClassifierCV(Isotonic Regression) 보정 적용
- 보정 전후 PR-AUC + Calibration Gap 비교하여 최종 모델 선정
- 비용함수(FN/FP) 기반 민감도 분석으로 최적 Threshold 도출

**진행 순서**
1. 데이터 + 변수 세트 불러오기 + LightGBM best params 로드 (day6_tuning_results.pkl)
2. 타겟 인코딩 함수 재정의 (커널 재부팅으로 인한 재정의 필요)
3. LightGBM(튜닝) + RF(기본값)에 CalibratedClassifierCV 적용 후 OOF 평가
4. 보정 전후 PR-AUC + Calibration Gap 비교
5. 최종 모델 선정 (PR-AUC 기준: LightGBM 0.5517 > RF 0.5100)
6. Threshold Optimization
   - FN 비용: 사기 거래 TransactionAmt 기반 추정 (중앙값 75.0, 75분위수 161.0, 평균 149.24)
   - FP 비용: 민감도 분석 (FP = 1, 3, 5, 10, 20, 30)
   - 총 18가지 시나리오 분석 → 대표 시나리오 확정 (threshold=0.20, FN:FP=7.5:1)

**최종 결과**
- 최종 모델: LightGBM (튜닝 + CalibratedClassifierCV Isotonic)
- 최종 threshold: 0.20 (중앙값+FP=10, FN:FP=7.5:1 기준)
- 보정 후 PR-AUC: 0.5517 (보정 전 0.5101 대비 +0.0416 향상)
- 보정 후 Calibration Gap: 0.1447 (보정 전 0.3130 대비 53.7% 개선)

### 1. 라이브러리 임포트 및 데이터 불러오기

In [1]:
import pandas as pd
import numpy as np
import pickle
import os
import sys
sys.path.append("..")

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import average_precision_score, brier_score_loss
from lightgbm import LGBMClassifier

pd.set_option('display.max_columns', 100)

# 데이터
df = pd.read_parquet("../data/processed/day3_final.parquet")

# LightGBM best params
with open("../data/processed/day6_tuning_results.pkl", 'rb') as f:
    tuning_results = pickle.load(f)

best_params_lgbm = tuning_results['best_params']['LightGBM']

# 변수 세트 재정의 (Day 4/6과 동일)
exclude_cols = ['TransactionID', 'isFraud', 'TransactionDT', 'UID', 'UID_v2']
target = 'isFraud'
groups = df['UID']

categorical_cols = df.drop(columns=exclude_cols).select_dtypes(include='category').columns.tolist()
numeric_cols = df.drop(columns=exclude_cols).select_dtypes(include=[np.number]).columns.tolist()
binary_cols = [c for c in numeric_cols if df[c].nunique() <= 2]
scale_cols = [c for c in numeric_cols if c not in binary_cols]

X = df.drop(columns=exclude_cols + [target])
y = df[target]

print(f"shape: {df.shape}")
print(f"LightGBM best params: {best_params_lgbm}")

c:\Users\seonu\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


shape: (590540, 208)
LightGBM best params: {'learning_rate': 0.1563037874293778, 'num_leaves': 165, 'min_child_samples': 32, 'subsample': 0.9248534606596772, 'colsample_bytree': 0.6651160762107319, 'reg_lambda': 0.00025077983643406634}


### 2. 사전 함수 재정의

Day 4/6에서 사용한 타겟 인코딩 함수를 재정의합니다.
커널 재부팅으로 인해 이전 노트북의 함수가 사라졌으므로 재정의가 필요합니다.

In [2]:
def target_encode(train_df, valid_df, cols, target_col, smoothing=20):
    global_mean = train_df[target_col].mean()
    encoded_train = train_df.copy()
    encoded_valid = valid_df.copy()
    for col in cols:
        encoded_train[col] = encoded_train[col].astype(str)
        encoded_valid[col] = encoded_valid[col].astype(str)
        stats = train_df.assign(**{col: train_df[col].astype(str)}).groupby(col)[target_col].agg(['mean', 'count'])
        smoothed = (stats['mean'] * stats['count'] + global_mean * smoothing) / (stats['count'] + smoothing)
        encoded_train[col] = encoded_train[col].map(smoothed).fillna(global_mean)
        encoded_valid[col] = encoded_valid[col].map(smoothed).fillna(global_mean)
    return encoded_train, encoded_valid

print("함수 정의 완료")

함수 정의 완료


### 3. CalibratedClassifierCV 개념 및 적용 방식

**CalibratedClassifierCV란**
모델이 출력하는 예측 확률을 실제 사기 발생 확률에 가깝게 보정하는 방법입니다.
내부적으로 CV를 사용하여 보정 모델을 학습하므로, 보정 과정에서 데이터 누수가 없습니다.

**보정 방법 두 가지**
- Platt Scaling (method='sigmoid'): 예측 확률에 Logistic Regression을 추가로 피팅. 간단하고 빠름
- Isotonic Regression (method='isotonic'): 단조 증가 함수로 보정. 데이터가 많을 때 더 유연하게 작동

**선택: Isotonic Regression**
우리 데이터는 59만 건으로 충분히 크므로 Isotonic Regression이 더 유연한 보정을 제공함.

**적용 방식**
- cv=5로 설정하여 5-fold CV 내부에서 보정 모델을 학습
- StratifiedGroupKFold가 아닌 일반 cv=5를 사용하는 이유:
  CalibratedClassifierCV는 보정 자체의 안정성을 위한 내부 CV이며,
  UID 그룹 분리는 외부 평가(Day 4 run_cv)에서 이미 적용했음

### 참고: Isotonic Regression을 선택한 이유

**Platt Scaling vs Isotonic Regression 비교**

| 구분 | Platt Scaling | Isotonic Regression |
|---|---|---|
| 보정 방식 | sigmoid(A × 원래확률 + B), S자 함수로 보정 | 단조 증가 조건만 만족하면 어떤 형태도 가능 |
| 유연성 | 낮음 (S자 형태로만 보정 가능) | 높음 (복잡한 왜곡 패턴도 보정 가능) |
| 필요 데이터 | 적어도 안정적 | 많을수록 안정적 (과적합 위험 있음) |
| 적합한 상황 | 왜곡이 단순하거나 데이터가 적을 때 | 왜곡이 크고 복잡하며 데이터가 충분할 때 |

**우리가 Isotonic Regression을 선택한 이유**

Day 5 Calibration 분석에서 확인한 Gap이 꽤 컸음:
- RF: 평균 Gap 0.1366 (전 구간 과소신)
- LightGBM: 평균 Gap 0.3130 (전 구간 과신, 중간 구간에서 0.5 이상)

이렇게 왜곡 패턴이 복잡하고 Gap이 큰 경우, Platt Scaling의 단순한 S자 함수로는 보정이 충분하지 않을 수 있음. Isotonic Regression의 유연한 보정이 더 효과적임.

또한 우리 데이터는 59만 건으로 충분히 크기 때문에, Isotonic Regression의 "데이터가 많아야 과적합 없이 안정적으로 작동한다"는 조건도 문제없이 충족함.

**면접 한 줄 설명**
"Calibration Gap이 크고 복잡한 패턴을 보여서 단순 S자 함수인 Platt Scaling보다 더 유연하게 보정할 수 있는 Isotonic Regression을 선택했고, 59만 건의 충분한 데이터가 있어 과적합 위험도 낮다고 판단했습니다."

In [3]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import average_precision_score, brier_score_loss
import warnings
warnings.filterwarnings('ignore')

imbalance_ratio = (y == 0).sum() / (y == 1).sum()

# 최종 후보 2개 모델 정의
model_lgbm = LGBMClassifier(
    **best_params_lgbm,
    n_estimators=5000,
    scale_pos_weight=imbalance_ratio,
    random_state=42,
    verbosity=-1,
)

model_rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42,
)

# CalibratedClassifierCV 적용
calibrated_lgbm = CalibratedClassifierCV(model_lgbm, method='isotonic', cv=5)
calibrated_rf   = CalibratedClassifierCV(model_rf,   method='isotonic', cv=5)

print("모델 정의 완료")
print(f"LightGBM best params: {best_params_lgbm}")
print(f"클래스 불균형 비율: 1:{imbalance_ratio:.1f}")

모델 정의 완료
LightGBM best params: {'learning_rate': 0.1563037874293778, 'num_leaves': 165, 'min_child_samples': 32, 'subsample': 0.9248534606596772, 'colsample_bytree': 0.6651160762107319, 'reg_lambda': 0.00025077983643406634}
클래스 불균형 비율: 1:27.6


### 4. 보정 전후 비교를 위한 OOF 평가

보정 전후 성능을 공정하게 비교하기 위해,
StratifiedGroupKFold 5-fold OOF 방식으로 보정된 모델의 PR-AUC와 Calibration Gap을 측정합니다.

보정 전 결과(Day 4/5)와 동일한 조건에서 비교합니다.
- LightGBM 보정 전: PR-AUC 0.4860 (기본값), 0.5101 (튜닝 후)
- RF 보정 전: PR-AUC 0.4972 (기본값)

In [4]:
def evaluate_calibrated_model(model, model_name, use_category=False):
    """보정된 모델의 OOF PR-AUC + Calibration Gap 측정"""
    sgkf = StratifiedGroupKFold(n_splits=5)
    oof_preds  = np.zeros(len(df))
    oof_labels = np.zeros(len(df))
    pr_aucs    = []

    for fold, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups=groups)):
        X_train = X.iloc[train_idx].copy()
        X_val   = X.iloc[val_idx].copy()
        y_train = y.iloc[train_idx]
        y_val   = y.iloc[val_idx]

        if not use_category:
            temp_train = pd.concat([X_train[categorical_cols], y_train], axis=1)
            temp_val   = X_val[categorical_cols].copy()
            temp_train_enc, temp_val_enc = target_encode(
                temp_train, temp_val, categorical_cols, target
            )
            X_train[categorical_cols] = temp_train_enc[categorical_cols].values
            X_val[categorical_cols]   = temp_val_enc[categorical_cols].values

        scaler = StandardScaler()
        X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
        X_val[scale_cols]   = scaler.transform(X_val[scale_cols])

        if use_category:
            for col in categorical_cols:
                X_train[col] = X_train[col].astype('category')
                X_val[col]   = X_val[col].astype('category')
        else:
            for col in categorical_cols:
                X_train[col] = X_train[col].astype(float)
                X_val[col]   = X_val[col].astype(float)

        model.fit(X_train, y_train)
        y_pred = model.predict_proba(X_val)[:, 1]

        oof_preds[val_idx]  = y_pred
        oof_labels[val_idx] = y_val.values

        pr_auc = average_precision_score(y_val, y_pred)
        pr_aucs.append(pr_auc)
        print(f"  [Fold {fold+1}] PR-AUC: {pr_auc:.4f}")

    # Calibration Gap 계산
    prob_true, prob_pred = calibration_curve(
        oof_labels, oof_preds, n_bins=10, strategy='uniform'
    )
    gap = np.abs(prob_true - prob_pred)
    brier = brier_score_loss(oof_labels, oof_preds)

    print(f"\n  [{model_name}] 평균 PR-AUC: {np.mean(pr_aucs):.4f} (±{np.std(pr_aucs):.4f})")
    print(f"  [{model_name}] 평균 |Gap|: {gap.mean():.4f} | Brier Score: {brier:.4f}")

    return {
        'model_name':  model_name,
        'pr_auc_mean': np.mean(pr_aucs),
        'pr_auc_std':  np.std(pr_aucs),
        'mean_gap':    gap.mean(),
        'brier':       brier,
        'oof_preds':   oof_preds,
        'oof_labels':  oof_labels,
    }


print("=== LightGBM (튜닝 + 보정) ===")
result_lgbm = evaluate_calibrated_model(calibrated_lgbm, 'LightGBM(보정)', use_category=True)

print("\n=== Random Forest (기본값 + 보정) ===")
result_rf = evaluate_calibrated_model(calibrated_rf, 'RF(보정)', use_category=False)

=== LightGBM (튜닝 + 보정) ===
  [Fold 1] PR-AUC: 0.5472
  [Fold 2] PR-AUC: 0.5537
  [Fold 3] PR-AUC: 0.5581
  [Fold 4] PR-AUC: 0.5070
  [Fold 5] PR-AUC: 0.5923

  [LightGBM(보정)] 평균 PR-AUC: 0.5517 (±0.0272)
  [LightGBM(보정)] 평균 |Gap|: 0.1447 | Brier Score: 0.0241

=== Random Forest (기본값 + 보정) ===
  [Fold 1] PR-AUC: 0.5157
  [Fold 2] PR-AUC: 0.5125
  [Fold 3] PR-AUC: 0.5154
  [Fold 4] PR-AUC: 0.4504
  [Fold 5] PR-AUC: 0.5561

  [RF(보정)] 평균 PR-AUC: 0.5100 (±0.0339)
  [RF(보정)] 평균 |Gap|: 0.0507 | Brier Score: 0.0231


In [5]:
import pickle, os

calibration_results = {
    'lgbm_calibrated': {
        'pr_auc_mean': result_lgbm['pr_auc_mean'],
        'pr_auc_std': result_lgbm['pr_auc_std'],
        'mean_gap': result_lgbm['mean_gap'],
        'brier': result_lgbm['brier'],
        'oof_preds': result_lgbm['oof_preds'],
        'oof_labels': result_lgbm['oof_labels'],
    },
    'rf_calibrated': {
        'pr_auc_mean': result_rf['pr_auc_mean'],
        'pr_auc_std': result_rf['pr_auc_std'],
        'mean_gap': result_rf['mean_gap'],
        'brier': result_rf['brier'],
        'oof_preds': result_rf['oof_preds'],
        'oof_labels': result_rf['oof_labels'],
    }
}

with open("../data/processed/day7_calibration_results.pkl", 'wb') as f:
    pickle.dump(calibration_results, f)

print("저장 완료")
print(f"LightGBM(보정): PR-AUC={result_lgbm['pr_auc_mean']:.4f}, Gap={result_lgbm['mean_gap']:.4f}")
print(f"RF(보정): PR-AUC={result_rf['pr_auc_mean']:.4f}, Gap={result_rf['mean_gap']:.4f}")

저장 완료
LightGBM(보정): PR-AUC=0.5517, Gap=0.1447
RF(보정): PR-AUC=0.5100, Gap=0.0507


### 5. 보정 전후 비교 결과 해석

**보정 전후 PR-AUC 비교**

| 모델 | 보정 전 PR-AUC | 보정 후 PR-AUC | 향상 | 보정 후 Gap |
|---|---|---|---|---|
| LightGBM | 0.5101 (튜닝 후) | **0.5517** | +0.0416 | 0.1447 |
| RF | 0.4972 (기본값) | **0.5100** | +0.0128 | 0.0507 |

**주요 관찰**

1. **보정이 PR-AUC를 크게 향상시킴**
   - LightGBM: +0.0416으로 가장 큰 향상. 보정 전 Calibration Gap이 0.3130으로 가장 컸던 모델이라, 보정 효과도 가장 크게 나타남
   - RF: +0.0128로 상대적으로 향상 폭이 작음. 보정 전에도 Gap이 0.1366으로 상대적으로 양호했기 때문

2. **보정 후 PR-AUC 순위 역전**
   - 보정 전: RF(0.4972) > LightGBM 튜닝(없음, 기본값 기준) → 엄밀히는 LightGBM 튜닝(0.5101) > RF(0.4972)
   - 보정 후: LightGBM(0.5517) > RF(0.5100)
   - 보정 효과가 LightGBM에서 더 크게 나타나 격차가 벌어짐

3. **Calibration 품질은 RF가 여전히 우위**
   - RF 보정 후 Gap: 0.0507 (양호)
   - LightGBM 보정 후 Gap: 0.1447 (여전히 RF보다 나쁨)
   - 보정이 LightGBM의 Calibration을 개선(0.3130 → 0.1447)했지만, RF 수준(0.0507)에는 미치지 못함

**PR-AUC vs Calibration 관점 정리**

| 기준 | 우위 모델 | 수치 |
|---|---|---|
| PR-AUC (사기 탐지 성능) | LightGBM | 0.5517 vs 0.5100 (+0.0417) |
| Calibration (확률 신뢰도) | RF | Gap 0.0507 vs 0.1447 |
| Brier Score | RF | 0.0231 vs 0.0241 |

**결론 방향**
- 사기 탐지 성능(PR-AUC) 기준: LightGBM 우위
- 확률 신뢰도(Calibration) 기준: RF 우위
- 두 기준이 충돌하므로, 최종 모델 선정은 "비즈니스 목적"을 기준으로 판단해야 함
  - 사기를 최대한 많이 잡는 것이 목적이면 → LightGBM
  - 확률값 자체의 신뢰도가 중요한 운영 환경이면 → RF

### 6. 최종 모델 선정

**최종 모델: LightGBM (튜닝 + CalibratedClassifierCV 보정)**

**선정 근거**

본 프로젝트는 기획안 제목("비용 민감형 사기탐지")에서 명시한 대로, FN(사기를 놓침)의 비용이 FP(고객 불편)보다 훨씬 크다는 전제 하에 설계됨. 따라서 "최대한 많이 잡는 것"이 1차 목표이며, 이를 반영한 핵심 지표로 PR-AUC를 선정하였음(Day 4에서 명시적으로 결정).

이 기준에서 LightGBM(보정 후 PR-AUC 0.5517)이 RF(보정 후 PR-AUC 0.5100)보다 우위이므로 LightGBM을 최종 모델로 확정함.

**보정 후 최종 비교**

| 기준 | LightGBM (최종 선택) | RF |
|---|---|---|
| PR-AUC (핵심 지표) | **0.5517** | 0.5100 |
| Calibration Gap | 0.1447 | **0.0507** |
| Brier Score | 0.0241 | **0.0231** |

**RF 탈락 이유**
- Calibration 품질은 RF가 우위이나, 본 프로젝트의 핵심 목표(사기를 최대한 잡는 것)에서 PR-AUC가 더 낮아 탈락
- RF는 "Calibration이 더 좋은 대안 모델로 검토했으나, PR-AUC 기준 LightGBM에 밀려 탈락"이라는 의사결정 과정 자체가 분석의 엄밀성을 보여주는 근거로 활용됨

**LightGBM Calibration 보정 효과**
- 보정 전 Gap: 0.3130 → 보정 후 Gap: 0.1447 (53.7% 개선)
- 보정 전 PR-AUC: 0.5101 → 보정 후 PR-AUC: 0.5517 (+0.0416)
- 보정 후 Gap 0.1447은 RF(0.0507) 수준에는 미치지 못하나, 4.9절 Threshold Optimization을 수행하기에 충분한 수준으로 판단함

**최종 확정**
- 최종 모델: LightGBM (튜닝 파라미터 + Isotonic Regression 보정)
- 다음 단계: 보정된 LightGBM의 예측 확률을 기반으로 비용함수(FP/FN) 기반 Threshold Optimization 수행

---

### 7. Threshold Optimization 준비 — 저장된 보정 결과 불러오기

커널 재부팅으로 인해 저장된 보정 결과(day7_calibration_results.pkl)를 불러옵니다.
LightGBM(보정) OOF 예측값을 기반으로 Threshold Optimization을 진행합니다.

In [1]:
import pandas as pd
import numpy as np
import pickle

with open("../data/processed/day7_calibration_results.pkl", 'rb') as f:
    calibration_results = pickle.load(f)

oof_preds  = calibration_results['lgbm_calibrated']['oof_preds']
oof_labels = calibration_results['lgbm_calibrated']['oof_labels'].astype(int)

print(f"OOF 예측값 shape: {oof_preds.shape}")
print(f"OOF 실제값 shape: {oof_labels.shape}")
print(f"사기 비율: {oof_labels.mean():.4f}")
print(f"예측값 범위: {oof_preds.min():.4f} ~ {oof_preds.max():.4f}")

print(f"\n[확인] LightGBM(보정) PR-AUC: {calibration_results['lgbm_calibrated']['pr_auc_mean']:.4f}")
print(f"[확인] LightGBM(보정) Gap: {calibration_results['lgbm_calibrated']['mean_gap']:.4f}")

OOF 예측값 shape: (590540,)
OOF 실제값 shape: (590540,)
사기 비율: 0.0350
예측값 범위: 0.0019 ~ 0.9935

[확인] LightGBM(보정) PR-AUC: 0.5517
[확인] LightGBM(보정) Gap: 0.1447


### 8. Threshold Optimization

**목표**
보정된 LightGBM의 예측 확률을 기반으로, 비용함수(FP/FN 비용)를 최소화하는 최적 임계값을 탐색합니다.

**비용 설정 방식**

- FN 비용: 사기 거래의 평균 TransactionAmt로 추정 (데이터 기반)
  - "사기를 놓치면 평균적으로 그 거래금액만큼 손실이 발생한다"는 가정
- FP 비용: 수작업 검토, 추가 인증, CS 대응 등 운영비용으로 간주
  - 실제 운영비용 데이터가 없으므로 민감도 분석으로 처리 (FP 비용 = 1, 3, 5)

**진행 순서**

1. FN 비용 추정 (사기 거래 평균 TransactionAmt)
2. 임계값 범위(0.01~0.99) 전체 탐색
3. FP 비용 시나리오별(1, 3, 5) 최적 threshold 도출
4. 각 시나리오별 precision, recall, F1, confusion matrix 확인
5. threshold 안정성 검증 (시나리오별 결과가 크게 달라지지 않으면 robust한 것으로 판단)

**참고: 실제 운영 비용 데이터가 없는 경우의 처리 방침**

FP 비용은 서비스별로 크게 달라지므로(카드 결제, 간편결제, 대출 등), 
본 프로젝트에서는 상대비율 기반 민감도 분석으로 threshold를 결정하며,
실제 운영 환경에서는 업무 프로세스 기준 FP 비용이 구체화되면 재조정하는 것을 권장함.

### 8-1. FN 비용 추정 (사기 거래 평균 TransactionAmt)

In [2]:
import pandas as pd
import numpy as np

df = pd.read_parquet("../data/processed/day3_final.parquet")

fraud_transactions = df[df['isFraud'] == 1]['TransactionAmt']
fn_cost = fraud_transactions.mean()

print(f"사기 거래 건수: {len(fraud_transactions):,}건")
print(f"사기 거래 TransactionAmt 기초통계:")
print(fraud_transactions.describe())
print(f"\nFN 비용 (평균 TransactionAmt): {fn_cost:.2f}")
print(f"\n민감도 분석 시나리오:")
for fp_cost in [1, 3, 5]:
    print(f"  FP={fp_cost} → FN:FP 비율 = {fn_cost/fp_cost:.1f}:1")

사기 거래 건수: 20,663건
사기 거래 TransactionAmt 기초통계:
count    20663.000000
mean       149.244766
std        232.212158
min          0.292000
25%         35.043999
50%         75.000000
75%        161.000000
max       5191.000000
Name: TransactionAmt, dtype: float64

FN 비용 (평균 TransactionAmt): 149.24

민감도 분석 시나리오:
  FP=1 → FN:FP 비율 = 149.2:1
  FP=3 → FN:FP 비율 = 49.7:1
  FP=5 → FN:FP 비율 = 29.8:1


### 8-2. FN 비용 후보 설정 및 민감도 분석 시나리오 확정

**FN 비용 후보 3가지**
- 중앙값(75.0): 보수적 기준, "전형적인 사기 1건 손실" 대표값
  분포가 오른쪽으로 치우치고 극단값이 드문 경우 가장 안정적인 기준
- 75분위수(161.0): 중간 수준, 상위 25% 손실을 일부 반영한 기준
- 평균(149.24): 극단값 포함 전체 손실 반영, 소수 고액 사기까지 고려

**FP 비용 시나리오 3가지**
- FP=1, 3, 5 (수작업 검토, 추가 인증, CS 대응 등 운영비용의 FN 비용과 같은 단위의 FP 비용 가정값)

**전체 시나리오: 3(FN 기준) × 3(FP 시나리오) = 9가지**
각 시나리오별 최적 threshold를 도출하고, 결과가 안정적인지(robust) 검증함.

**보고서 표현**
"FN 비용은 사기 거래 TransactionAmt의 중앙값(75.0), 75분위수(161.0), 평균(149.24)을
각각 기준으로 추정하였으며, FP 비용은 운영비용 가정값(1, 3, 5)으로 설정하여
총 9가지 시나리오에 대한 민감도 분석을 수행하였다."

In [3]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

fn_cost_candidates = {
    '중앙값(75.0)':     75.0,
    '75분위수(161.0)': 161.0,
    '평균(149.24)':    149.24,
}
fp_cost_candidates = [1, 3, 5]
thresholds = np.arange(0.01, 1.0, 0.01)

scenario_results = []

for fn_name, fn_cost in fn_cost_candidates.items():
    for fp_cost in fp_cost_candidates:
        best_threshold = None
        best_total_cost = np.inf

        for threshold in thresholds:
            y_pred_binary = (oof_preds >= threshold).astype(int)
            tn, fp, fn, tp = confusion_matrix(oof_labels, y_pred_binary).ravel()

            total_cost = fn * fn_cost + fp * fp_cost

            if total_cost < best_total_cost:
                best_total_cost = total_cost
                best_threshold = threshold
                best_tn, best_fp, best_fn, best_tp = tn, fp, fn, tp

        y_pred_best = (oof_preds >= best_threshold).astype(int)
        precision = precision_score(oof_labels, y_pred_best)
        recall    = recall_score(oof_labels, y_pred_best)
        f1        = f1_score(oof_labels, y_pred_best)

        scenario_results.append({
            'FN_기준':     fn_name,
            'FP_비용':     fp_cost,
            'FN:FP 비율':  f"{fn_cost/fp_cost:.1f}:1",
            '최적 threshold': best_threshold,
            'Precision':   round(precision, 4),
            'Recall':      round(recall, 4),
            'F1':          round(f1, 4),
            'FN(놓친사기)': best_fn,
            'FP(오탐)':    best_fp,
            '총 비용':     round(best_total_cost, 1),
        })

scenario_df = pd.DataFrame(scenario_results)
print("=== 9가지 시나리오별 최적 Threshold 결과 ===")
print(scenario_df.to_string(index=False))

=== 9가지 시나리오별 최적 Threshold 결과 ===
       FN_기준  FP_비용 FN:FP 비율  최적 threshold  Precision  Recall     F1  FN(놓친사기)  FP(오탐)      총 비용
   중앙값(75.0)      1   75.0:1          0.05     0.0847  0.8379 0.1538      3349  187155  438330.0
   중앙값(75.0)      3   25.0:1          0.12     0.1883  0.6967 0.2964      6268   62072  656316.0
   중앙값(75.0)      5   15.0:1          0.16     0.2781  0.6313 0.3861      7619   33863  740740.0
75분위수(161.0)      1  161.0:1          0.01     0.0371  0.9959 0.0715        85  534356  548041.0
75분위수(161.0)      3   53.7:1          0.06     0.0988  0.8096 0.1760      3934  152671 1091387.0
75분위수(161.0)      5   32.2:1          0.10     0.1526  0.7334 0.2527      5509   84119 1307544.0
  평균(149.24)      1  149.2:1          0.01     0.0371  0.9959 0.0715        85  534356  547041.4
  평균(149.24)      3   49.7:1          0.07     0.1127  0.7856 0.1972      4430  127742 1044359.2
  평균(149.24)      5   29.8:1          0.11     0.1730  0.7116 0.2784      5959   70274 124069

### 8-3. 민감도 분석 결과 해석

**9가지 시나리오 결과 요약**

| FN 기준 | FP 비용 | FN:FP 비율 | 최적 threshold | Recall | Precision | FP(오탐) |
|---|---|---|---|---|---|---|
| 중앙값(75.0) | 1 | 75:1 | 0.05 | 0.838 | 0.085 | 187,155 |
| 중앙값(75.0) | 3 | 25:1 | 0.12 | 0.697 | 0.188 | 62,072 |
| 중앙값(75.0) | 5 | 15:1 | 0.16 | 0.631 | 0.278 | 33,863 |
| 75분위수(161.0) | 1 | 161:1 | 0.01 | 0.996 | 0.037 | 534,356 |
| 75분위수(161.0) | 3 | 54:1 | 0.06 | 0.810 | 0.099 | 152,671 |
| 75분위수(161.0) | 5 | 32:1 | 0.10 | 0.733 | 0.153 | 84,119 |
| 평균(149.24) | 1 | 149:1 | 0.01 | 0.996 | 0.037 | 534,356 |
| 평균(149.24) | 3 | 50:1 | 0.07 | 0.786 | 0.113 | 127,742 |
| 평균(149.24) | 5 | 30:1 | 0.11 | 0.712 | 0.173 | 70,274 |

**주요 관찰**

1. **FN:FP 비율이 높을수록 threshold가 낮아지고 Recall이 높아짐**
   - 75분위수/평균 + FP=1 조합(FN:FP ≥ 149:1)은 threshold=0.01, Recall=0.996이지만
     FP(오탐)가 53만 건으로 운영상 감당 불가능한 수준
   - 사기를 최대한 잡으려 할수록 정상 거래도 함께 막아버리는 trade-off가 명확하게 나타남

2. **중앙값 기준이 가장 현실적인 운영 시나리오를 제공함**
   - 중앙값+FP=3 (threshold=0.12): Recall=0.697, FP=62,072건
   - 중앙값+FP=5 (threshold=0.16): Recall=0.631, FP=33,863건
   - 두 시나리오가 "사기를 어느 정도 잡으면서 오탐도 감당 가능한 수준"에 해당함

3. **Threshold 안정성 확인 (Robust)**
   - FP=3 시나리오: FN 기준이 달라져도 threshold가 0.06~0.12 범위로 수렴
   - FP=5 시나리오: threshold가 0.10~0.16 범위로 수렴
   - FN 비용 추정값이 다소 달라져도 최적 threshold가 크게 변하지 않음 → 결론이 robust함

4. **Precision이 전반적으로 낮음**
   - 대부분 Precision이 0.04~0.28 수준으로, threshold를 낮게 잡을수록 오탐이 많아짐
   - 이는 사기 비율(3.5%)이 낮은 불균형 데이터 특성과,
     FN이 훨씬 비싼 비용 구조가 threshold를 낮게 유도하는 것에 기인함

**최종 권장 시나리오**
- 운영 현실성을 고려했을 때 **중앙값+FP=3 (threshold=0.12)** 또는
  **중앙값+FP=5 (threshold=0.16)** 이 가장 균형 잡힌 선택임
- 실제 운영 환경에서 FP 처리 가능 건수(운영 부서 수용 가능성)에 따라
  최종 threshold를 조정하는 것을 권장함
- 본 분석에서는 **threshold=0.12(중앙값+FP=3)를 대표 시나리오로 채택**하여
  이후 최종 성능 보고에 활용함

### 참고: FP 비용 범위 확장 이유

**기존 설정(FP=1,3,5)과 실무 참고값(FN:FP=5:1~10:1)의 차이**

처음에 "실무에서 FN:FP 비율을 10:1 또는 5:1로 설정하는 경우가 많다"고 언급했으나,
기존 시나리오(FP=1,3,5) 결과는 FN:FP 비율이 25:1~161:1로 나타나 실무 참고값과 차이가 있음.

**원인**
- FP 비용(1,3,5달러)이 실제 운영비용(수작업 검토, CS 대응, 고객 이탈 등)보다
  낮게 설정되었을 가능성이 있음
- 실제 FP 운영비용이 10~30달러 수준이라면 FN:FP 비율이 5:1~15:1로 떨어져
  실무 참고값과 유사해짐

**FP 비용 범위 확장**
- 기존: FP = 1, 3, 5
- 추가: FP = 10, 20, 30
- FN:FP 비율 커버 범위:

| FP 비용 | 중앙값(75) 기준 | 평균(149.24) 기준 |
|---|---|---|
| 1 | 75:1 | 149:1 |
| 3 | 25:1 | 50:1 |
| 5 | 15:1 | 30:1 |
| 10 | 7.5:1 | 14.9:1 |
| 20 | 3.75:1 | 7.5:1 |
| 30 | 2.5:1 | 5:1 |

FP=10~30 범위를 추가하면 실무 참고값(5:1~10:1)까지 커버 가능함.

### 8-4. FP 비용 범위 확장 민감도 분석 (FP = 1, 3, 5, 10, 20, 30)

기존 시나리오(FP=1,3,5)에 FP=10,20,30을 추가하여 총 18가지 시나리오로 확장합니다.
FN:FP 비율 2.5:1~161:1 범위를 커버하여 실무 참고값(5:1~10:1)까지 포함합니다.

In [4]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

fn_cost_candidates = {
    '중앙값(75.0)':     75.0,
    '75분위수(161.0)': 161.0,
    '평균(149.24)':    149.24,
}
fp_cost_candidates = [1, 3, 5, 10, 20, 30]
thresholds = np.arange(0.01, 1.0, 0.01)

scenario_results_v2 = []

for fn_name, fn_cost in fn_cost_candidates.items():
    for fp_cost in fp_cost_candidates:
        best_threshold = None
        best_total_cost = np.inf

        for threshold in thresholds:
            y_pred_binary = (oof_preds >= threshold).astype(int)
            tn, fp, fn, tp = confusion_matrix(oof_labels, y_pred_binary).ravel()
            total_cost = fn * fn_cost + fp * fp_cost

            if total_cost < best_total_cost:
                best_total_cost = total_cost
                best_threshold = threshold
                best_tn, best_fp, best_fn, best_tp = tn, fp, fn, tp

        y_pred_best = (oof_preds >= best_threshold).astype(int)
        precision = precision_score(oof_labels, y_pred_best)
        recall    = recall_score(oof_labels, y_pred_best)
        f1        = f1_score(oof_labels, y_pred_best)

        scenario_results_v2.append({
            'FN_기준':          fn_name,
            'FP_비용':          fp_cost,
            'FN:FP 비율':       f"{fn_cost/fp_cost:.1f}:1",
            '최적 threshold':   best_threshold,
            'Precision':        round(precision, 4),
            'Recall':           round(recall, 4),
            'F1':               round(f1, 4),
            'FN(놓친사기)':     best_fn,
            'FP(오탐)':         best_fp,
            '총 비용':          round(best_total_cost, 1),
        })

scenario_df_v2 = pd.DataFrame(scenario_results_v2)
print("=== 18가지 시나리오별 최적 Threshold 결과 ===")
print(scenario_df_v2.to_string(index=False))

=== 18가지 시나리오별 최적 Threshold 결과 ===
       FN_기준  FP_비용 FN:FP 비율  최적 threshold  Precision  Recall     F1  FN(놓친사기)  FP(오탐)      총 비용
   중앙값(75.0)      1   75.0:1          0.05     0.0847  0.8379 0.1538      3349  187155  438330.0
   중앙값(75.0)      3   25.0:1          0.12     0.1883  0.6967 0.2964      6268   62072  656316.0
   중앙값(75.0)      5   15.0:1          0.16     0.2781  0.6313 0.3861      7619   33863  740740.0
   중앙값(75.0)     10    7.5:1          0.20     0.4008  0.5608 0.4675      9076   17324  853940.0
   중앙값(75.0)     20    3.8:1          0.28     0.6239  0.4562 0.5270     11237    5682  956415.0
   중앙값(75.0)     30    2.5:1          0.33     0.7477  0.4114 0.5307     12163    2868  998265.0
75분위수(161.0)      1  161.0:1          0.01     0.0371  0.9959 0.0715        85  534356  548041.0
75분위수(161.0)      3   53.7:1          0.06     0.0988  0.8096 0.1760      3934  152671 1091387.0
75분위수(161.0)      5   32.2:1          0.10     0.1526  0.7334 0.2527      5509   84119 13075

### 8-5. FP 비용 범위 확장 민감도 분석 결과 해석

**핵심 관찰**

1. **FP 비용이 높아질수록 threshold가 올라가고 Precision이 높아짐**
   - FP 비용이 높아지면 오탐 1건의 손실이 커지므로, 모델이 오탐을 더 강하게 억제하는
     방향으로 threshold가 올라감
   - 중앙값 기준: FP=1(threshold 0.05, Precision 0.085) → FP=10(threshold 0.20, Precision 0.401)
     → FP=30(threshold 0.33, Precision 0.748)으로 threshold와 Precision이 함께 상승함
   - 반대로 Recall은 FP=1(0.838) → FP=10(0.561) → FP=30(0.411)으로 하락하여,
     오탐 억제와 사기 탐지 간의 trade-off가 명확하게 나타남

2. **실무 참고값(FN:FP 5:1~10:1) 범위에서의 결과**
   - 중앙값+FP=10 (7.5:1): threshold=0.20, Recall=0.561, Precision=0.401, FP=17,324건
   - 중앙값+FP=20 (3.8:1): threshold=0.28, Recall=0.456, Precision=0.624, FP=5,682건
   - 평균+FP=20 (7.5:1): threshold=0.20, Recall=0.561, Precision=0.401, FP=17,324건
   - 평균+FP=30 (5.0:1): threshold=0.23, Recall=0.513, Precision=0.495, FP=10,806건

3. **Threshold 안정성 검증 (매우 Robust)**
   - FN 기준이 달라져도 FN:FP 비율이 같으면 threshold가 거의 동일하게 나타남
     - 중앙값+FP=10(7.5:1) vs 평균+FP=20(7.5:1): 둘 다 threshold=0.20, 결과 완전 동일
     - 중앙값+FP=20(3.8:1) vs 75분위수+FP=30(5.4:1): threshold 0.28 vs 0.23으로 유사
   - FN 비용 추정 방식이 달라져도 같은 비율이면 결론이 바뀌지 않음 → 분석이 robust함

4. **운영 현실성 기준 권장 구간**

   | FP 비용 구간 | 오탐(FP) 건수 | 운영 현실성 |
   |---|---|---|
   | FP=1~3 | 6만~53만 건 | ❌ 운영 불가 수준 |
   | FP=5~10 | 1.7만~8.4만 건 | ⚠️ 운영 부담 크나 가능한 수준 |
   | FP=20~30 | 2,868~2만 건 | ✅ 현실적으로 감당 가능한 수준 |

**최종 권장 시나리오**
- 운영 현실성과 사기 탐지 성능의 균형을 고려했을 때
  **중앙값+FP=10 (threshold=0.20)** 또는 **중앙값+FP=20 (threshold=0.28)** 이 가장 적합함
- 실제 운영 환경에서 FP 처리 가능 건수(운영 부서 수용 가능성)에 따라 최종 결정을 권장함
- 본 분석에서는 **threshold=0.20 (중앙값+FP=10, FN:FP=7.5:1)을 대표 시나리오로 채택**
  - 실무 참고값(5:1~10:1) 범위 내에 속함
  - Recall과 Precision의 균형이 무너뜨리지 않음.
  - Recall 0.561로 사기의 절반 이상을 탐지
  - FP 17,324건으로 운영 부담이 있으나 감당 가능한 수준

> “비용비율 기반 민감도 분석 결과, 동일한 FN:FP 비율에서는 FN 절대값 정의가 달라져도 threshold가 거의 동일하게 수렴했으며, 운영 현실성과 탐지 성능의 균형을 고려해 threshold=0.20을 대표 임계값으로 채택했다.”

### 9. Day 7 산출물 저장

Threshold Optimization 결과와 최종 모델 선정 결과를 저장합니다.
이후 08(SHAP), 09(Drift) 노트북에서 참조합니다.

In [5]:
import pickle, os

day7_results = {
    'final_model': 'LightGBM (튜닝 + CalibratedClassifierCV Isotonic)',
    'final_threshold': 0.20,
    'final_scenario': '중앙값+FP=10, FN:FP=7.5:1',
    'threshold_scenarios': scenario_df_v2,
    'calibration_summary': {
        'lgbm': {
            'pr_auc_before': 0.5101,
            'pr_auc_after':  0.5517,
            'gap_before':    0.3130,
            'gap_after':     0.1447,
        },
        'rf': {
            'pr_auc_before': 0.4972,
            'pr_auc_after':  0.5100,
            'gap_before':    0.1366,
            'gap_after':     0.0507,
        }
    }
}

os.makedirs("../data/processed", exist_ok=True)
with open("../data/processed/day7_results.pkl", 'wb') as f:
    pickle.dump(day7_results, f)

print("저장 완료")
print(f"최종 모델: {day7_results['final_model']}")
print(f"최종 threshold: {day7_results['final_threshold']}")
print(f"대표 시나리오: {day7_results['final_scenario']}")

저장 완료
최종 모델: LightGBM (튜닝 + CalibratedClassifierCV Isotonic)
최종 threshold: 0.2
대표 시나리오: 중앙값+FP=10, FN:FP=7.5:1


### 10. 최종 Threshold 기준 성능 보고

대표 시나리오(threshold=0.20, 중앙값+FP=10, FN:FP=7.5:1) 기준의 최종 성능을 정리합니다.

### 최종 모델 성능 요약

**모델**: LightGBM (튜닝 + CalibratedClassifierCV Isotonic 보정)
**최종 Threshold**: 0.20 (중앙값+FP=10, FN:FP=7.5:1 기준)

**최종 성능 지표**

| 지표 | 값 |
|---|---|
| PR-AUC (보정 후) | 0.5517 |
| Threshold | 0.20 |
| Precision | 0.4008 |
| Recall | 0.5608 |
| F1 | 0.4675 |
| FN (놓친 사기) | 9,076건 |
| FP (오탐) | 17,324건 |

**Confusion Matrix**

|  | 예측 정상 | 예측 사기 |
|---|---|---|
| **실제 정상** | 552,153 | 17,324 |
| **실제 사기** | 9,076 | 11,587 |

**해석**
- 전체 사기 거래 20,663건 중 11,587건(56.1%)을 탐지함
- 오탐(FP) 17,324건은 전체 정상 거래(569,477건)의 3.0% 수준
- 비용함수 기준(FN:FP=7.5:1)에서 총 기대비용이 최소화되는 운영 임계값임
- 실제 운영 환경에서 FP 처리 가능 건수에 따라 threshold를 0.28(FP=5,682건)로
  조정하는 것도 고려 가능함 (중앙값+FP=20 시나리오)